In [ ]:
!pip install lazypredict plotly

In [ ]:
import pandas as pd

from lazypredict.Supervised import LazyClassifier
from google.colab import drive
import nltk
from nltk.stem import WordNetLemmatizer
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer

from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizador = WordNetLemmatizer()

drive.mount('/content/drive')

df_master = pd.read_csv("/content/drive/MyDrive/data_office_products.csv")

df_sample = df_master.sample(n=5000, random_state=42).reset_index(drop=True)
#df_sample = df_master.groupby('target', group_keys=False).apply(lambda x: x.sample(n=1666, random_state=42)).reset_index(drop=True)

#sss = StratifiedShuffleSplit(n_splits=1, train_size=5000, random_state=42)
#for index_muestra, _ in sss.split(df_master, df_master['target']):
#    df_sample = df_master.iloc[index_muestra].reset_index(drop=True)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500000 entries, 0 to 1499999
Data columns (total 14 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   title_usuario      1499734 non-null  object 
 1   text               1499772 non-null  object 
 2   parent_asin        1500000 non-null  object 
 3   helpful_vote       1500000 non-null  int64  
 4   verified_purchase  1500000 non-null  bool   
 5   title_catalogo     1499910 non-null  object 
 6   average_rating     1500000 non-null  float64
 7   rating_number      1500000 non-null  int64  
 8   price              1500000 non-null  float64
 9   categories         1500000 non-null  object 
 10  target             1500000 non-null  int64  
 11  review_length      1500000 non-null  int64  
 12  word_count         1500000 non-null  int64  
 13  exclamation_count  1500000 non-null  int64  
dtypes: bool(1), float64(2), int64(6), object(5)
memory usage: 150.2+ MB


In [ ]:
def limpiar_y_lematizar(texto):
    palabras = str(texto).split()
    lemas = [lemmatizador.lemmatize(p.lower()) for p in palabras]
    return " ".join(lemas)

df_sample['text_procesado'] = df_sample['text'].apply(limpiar_y_lematizar)

In [ ]:
cols_num = ['price', 'average_rating', 'rating_number', 'helpful_vote']
col_texto = 'text_procesado'

preprocesador = ColumnTransformer(
    transformers=[
        ('nlp', TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1, 2)), col_texto),
        ('num', StandardScaler(), cols_num)
    ],
    remainder='drop'
)

In [ ]:
X_sparse = preprocesador.fit_transform(df_sample)
X = X_sparse.toarray() if hasattr(X_sparse, "toarray") else X_sparse
y = df_sample['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
modelos, predicciones = clf.fit(X_train, X_test, y_train, y_test)

In [ ]:
resultados = modelos.reset_index()

In [ ]:
fig_automl = px.bar(
    resultados,
    x='Model',
    y='F1 Score',
    color='Accuracy',
    title="Auditoría de Modelos: F1-Score vs Accuracy (LazyPredict)",
    labels={'Model': 'Algoritmo', 'F1 Score': 'Puntuación F1'},
    height=600
)

fig_automl.update_layout(xaxis={'categoryorder':'total descending'})
fig_automl.show()

In [ ]:
modelo_explicativo = LGBMClassifier(random_state=42, verbose=-1)
modelo_explicativo.fit(X_train, y_train)

LGBMClassifier(random_state=42, verbose=-1)

In [ ]:
nombres_nlp = preprocesador.named_transformers_['nlp'].get_feature_names_out()
nombres_totales = list(nombres_nlp) + cols_num

In [ ]:
importancias = modelo_explicativo.feature_importances_

df_importancias = pd.DataFrame({
    'Caracteristica': nombres_totales,
    'Importancia': importancias
})

In [ ]:
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False).head(20)

In [ ]:
fig_importancia = px.bar(
    df_importancias,
    x='Importancia',
    y='Caracteristica',
    orientation='h',
    title="Peso Estadístico de las Variables en la Satisfacción",
    color='Importancia',
    color_continuous_scale='Viridis',
    height=700
)
fig_importancia.update_layout(yaxis={'categoryorder':'total ascending'})
fig_importancia.show()